# Phase 6 high-dimensional point-law shard 08 of 16

**Predicted runtime: 117.2 minutes** (wall, N_WORKERS=2, 199 perms, VR filtration).
Each task checkpoints every 25 replications to `results/phase6_highdim_shards/`.

This fleet explores **ambient dimension d=10,20,40** and **n=50,500** (ecology: raw ~40 dims, PCA to 3-4).
Core families: iid_null (size), weak_barcode_null (translation), same_support_density,
same_square_four_atom_density (atoms), topology_alt (disk vs circle).
Additionally sparse/dense mean shifts and PCA-failure constructions where signal lives
in low-variance subspace (top 3 PCs are pure noise).

Methods: PointMMD (fixed 0.30 + median), Energy, MST, kNN k=1,5,10, SlicedW, C2ST log/rf,
RawBlockMMD (m=25, Hilbert-Gaussian bag), SC-B (m=25, VR d=0,1), Hybrid a=0.50.
SC-A omitted (expensive, abandoned). Rosenbaum only for n=50 (pooled n≤100).

**Preproc**: `raw` = test on d-dim clouds directly; `pca3`/`pca4` = pooled PCA to 3/4 dims
(fitted on pooled unlabeled points, valid under H0) then test. PCA-failure families use
noise_scale=1.5 (high-noise background) so pca3 discards signal.

**Runtime → Run all**, leave tab open. Each task downloads its parquet when done.
Collect all `phase6_highdim_*.parquet` into `results/phase6_highdim_shards/` and run:

```
python experiments/phase6_highdim_pointlaw_tournament.py --mode aggregate --input-dir results/phase6_highdim_shards
```

Design hash `401912b22817e1af` | FILTRATION=vr | seed_root=20260826


In [ ]:
import os

SHARD_ID = 8
N_SHARDS = 16
SEED_ROOT = 20260826
WALL_BUDGET_MIN = 450
N_WORKERS = 2
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
print({"SHARD_ID": SHARD_ID, "N_SHARDS": N_SHARDS, "WALL_BUDGET_MIN": WALL_BUDGET_MIN, "N_WORKERS": N_WORKERS})


In [ ]:
import os, shutil, sys, subprocess
REPO_DIR = "/content/Pointcloud_Equality_Testing"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
ret = os.system("git clone --depth 1 https://github.com/hugogobato/Pointcloud_Equality_Testing.git " + REPO_DIR)
print("clone exit", ret)
os.chdir(REPO_DIR)
ret = os.system("python -m pip install -q -e .")
print("pip install -e . exit", ret)
# Ensure results dir
os.makedirs("results/phase6_highdim_shards", exist_ok=True)
print("repo ready", REPO_DIR)


In [ ]:
%%bash
set -e
pip install -q numpy scipy scikit-learn scikit-fda matplotlib networkx==3.4.2 gudhi pot ripser 2>&1 | tail -20
pip install -q git+https://github.com/hugogobato/tcda_uq.git 2>&1 | tail -5
echo '--- install done ---'
# Verify imports
python -c "import gudhi, sklearn, ripser; print('gudhi', gudhi.__version__)"


In [ ]:
import os, sys
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
sys.path.insert(0, "/content/Pointcloud_Equality_Testing")
# Smoke test
import numpy as np
from experiments.phase6_highdim_pointlaw_tournament import make_cloud_pair_highdim, Cell
c0,c1 = make_cloud_pair_highdim("iid_null", 50, 50, seed=123, d=10)
print("cloud shapes", c0.shape, c1.shape)
# Test point MMD and block with VR (should use gudhi, not ripser)
from experiments.phase6_highdim_pointlaw_tournament import _run_one
cell = Cell(family="iid_null", n0=50, n1=50, m=25, d=10, preproc="raw", role="gating_null", description="test")
rows = _run_one((cell, 0, ("PointMMD-Gaussian","RawBlockMMD","SC-B"), 19, None))
print("smoke", [(r["method"], r["status"], round(float(r["pvalue"]),3)) for r in rows])
print("imports OK |", os.cpu_count(), "CPUs")


In [ ]:
import os, json, time
from pathlib import Path
import pandas as pd
from experiments.phase6_highdim_pointlaw_tournament import Cell, run_shard, DESIGN_HASH

# Fleet controls — first executable cell after smoke
SHARD_ID = 8
N_SHARDS = 16
SEED_ROOT = 20260826
WALL_BUDGET_MIN = 450
N_WORKERS = 2
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
print({"SHARD_ID": SHARD_ID, "N_SHARDS": N_SHARDS, "WALL_BUDGET_MIN": WALL_BUDGET_MIN, "N_WORKERS": N_WORKERS, "DESIGN_HASH": DESIGN_HASH})

TASKS = [
  {
    "task_id": "iid_null_n500_n1500_m25_d10_preproc_raw_rep200_224",
    "cell_id": "iid_null_n500_n1500_m25_d10_preproc_raw",
    "family": "iid_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "iid_null_n500_n1500_m25_d20_preproc_raw_rep100_124",
    "cell_id": "iid_null_n500_n1500_m25_d20_preproc_raw",
    "family": "iid_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "iid_null_n500_n1500_m25_d40_preproc_raw_rep0_24",
    "cell_id": "iid_null_n500_n1500_m25_d40_preproc_raw",
    "family": "iid_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "iid_null_n500_n1500_m25_d40_preproc_raw_rep400_424",
    "cell_id": "iid_null_n500_n1500_m25_d40_preproc_raw",
    "family": "iid_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "weak_barcode_null_n500_n1500_m25_d10_preproc_raw_rep300_324",
    "cell_id": "weak_barcode_null_n500_n1500_m25_d10_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "weak_barcode_null_n500_n1500_m25_d20_preproc_raw_rep200_224",
    "cell_id": "weak_barcode_null_n500_n1500_m25_d20_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "weak_barcode_null_n500_n1500_m25_d40_preproc_raw_rep100_124",
    "cell_id": "weak_barcode_null_n500_n1500_m25_d40_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_support_density_n500_n1500_m25_d10_preproc_raw_rep0_24",
    "cell_id": "same_support_density_n500_n1500_m25_d10_preproc_raw",
    "family": "same_support_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_support_density_n500_n1500_m25_d10_preproc_raw_rep400_424",
    "cell_id": "same_support_density_n500_n1500_m25_d10_preproc_raw",
    "family": "same_support_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_support_density_n500_n1500_m25_d20_preproc_raw_rep300_324",
    "cell_id": "same_support_density_n500_n1500_m25_d20_preproc_raw",
    "family": "same_support_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_support_density_n500_n1500_m25_d40_preproc_raw_rep200_224",
    "cell_id": "same_support_density_n500_n1500_m25_d40_preproc_raw",
    "family": "same_support_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_square_four_atom_density_n500_n1500_m25_d10_preproc_raw_rep100_124",
    "cell_id": "same_square_four_atom_density_n500_n1500_m25_d10_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_square_four_atom_density_n500_n1500_m25_d20_preproc_raw_rep0_24",
    "cell_id": "same_square_four_atom_density_n500_n1500_m25_d20_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_square_four_atom_density_n500_n1500_m25_d20_preproc_raw_rep400_424",
    "cell_id": "same_square_four_atom_density_n500_n1500_m25_d20_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "same_square_four_atom_density_n500_n1500_m25_d40_preproc_raw_rep300_324",
    "cell_id": "same_square_four_atom_density_n500_n1500_m25_d40_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "topology_alt_n500_n1500_m25_d10_preproc_raw_rep200_224",
    "cell_id": "topology_alt_n500_n1500_m25_d10_preproc_raw",
    "family": "topology_alt",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "topology_alt_n500_n1500_m25_d20_preproc_raw_rep100_124",
    "cell_id": "topology_alt_n500_n1500_m25_d20_preproc_raw",
    "family": "topology_alt",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "topology_alt_n500_n1500_m25_d40_preproc_raw_rep0_24",
    "cell_id": "topology_alt_n500_n1500_m25_d40_preproc_raw",
    "family": "topology_alt",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "topology_alt_n500_n1500_m25_d40_preproc_raw_rep400_424",
    "cell_id": "topology_alt_n500_n1500_m25_d40_preproc_raw",
    "family": "topology_alt",
    "n0": 500,
    "n1": 500,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block"
    ],
    "predicted_seconds": 450.0
  },
  {
    "task_id": "iid_null_n50_n150_m25_d20_preproc_raw_rep0_24",
    "cell_id": "iid_null_n50_n150_m25_d20_preproc_raw",
    "family": "iid_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "iid_null_n50_n150_m25_d20_preproc_raw_rep400_424",
    "cell_id": "iid_null_n50_n150_m25_d20_preproc_raw",
    "family": "iid_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "iid_null_n50_n150_m25_d40_preproc_raw_rep300_324",
    "cell_id": "iid_null_n50_n150_m25_d40_preproc_raw",
    "family": "iid_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "iid_null",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "weak_barcode_null_n50_n150_m25_d10_preproc_raw_rep200_224",
    "cell_id": "weak_barcode_null_n50_n150_m25_d10_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "weak_barcode_null_n50_n150_m25_d20_preproc_raw_rep100_124",
    "cell_id": "weak_barcode_null_n50_n150_m25_d20_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "weak_barcode_null_n50_n150_m25_d40_preproc_raw_rep0_24",
    "cell_id": "weak_barcode_null_n50_n150_m25_d40_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "weak_barcode_null_n50_n150_m25_d40_preproc_raw_rep400_424",
    "cell_id": "weak_barcode_null_n50_n150_m25_d40_preproc_raw",
    "family": "weak_barcode_null",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "weak_barcode_null",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_support_density_n50_n150_m25_d10_preproc_raw_rep300_324",
    "cell_id": "same_support_density_n50_n150_m25_d10_preproc_raw",
    "family": "same_support_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_support_density_n50_n150_m25_d20_preproc_raw_rep200_224",
    "cell_id": "same_support_density_n50_n150_m25_d20_preproc_raw",
    "family": "same_support_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_support_density_n50_n150_m25_d40_preproc_raw_rep100_124",
    "cell_id": "same_support_density_n50_n150_m25_d40_preproc_raw",
    "family": "same_support_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_support_density",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_square_four_atom_density_n50_n150_m25_d10_preproc_raw_rep0_24",
    "cell_id": "same_square_four_atom_density_n50_n150_m25_d10_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_square_four_atom_density_n50_n150_m25_d10_preproc_raw_rep400_424",
    "cell_id": "same_square_four_atom_density_n50_n150_m25_d10_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_square_four_atom_density_n50_n150_m25_d20_preproc_raw_rep300_324",
    "cell_id": "same_square_four_atom_density_n50_n150_m25_d20_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "same_square_four_atom_density_n50_n150_m25_d40_preproc_raw_rep200_224",
    "cell_id": "same_square_four_atom_density_n50_n150_m25_d40_preproc_raw",
    "family": "same_square_four_atom_density",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "same_square_four_atom_density",
    "rep_start": 200,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "topology_alt_n50_n150_m25_d10_preproc_raw_rep100_124",
    "cell_id": "topology_alt_n50_n150_m25_d10_preproc_raw",
    "family": "topology_alt",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 10,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 100,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "topology_alt_n50_n150_m25_d20_preproc_raw_rep0_24",
    "cell_id": "topology_alt_n50_n150_m25_d20_preproc_raw",
    "family": "topology_alt",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 0,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "topology_alt_n50_n150_m25_d20_preproc_raw_rep400_424",
    "cell_id": "topology_alt_n50_n150_m25_d20_preproc_raw",
    "family": "topology_alt",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 20,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 400,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  },
  {
    "task_id": "topology_alt_n50_n150_m25_d40_preproc_raw_rep300_324",
    "cell_id": "topology_alt_n50_n150_m25_d40_preproc_raw",
    "family": "topology_alt",
    "n0": 50,
    "n1": 50,
    "m": 25,
    "d": 40,
    "preproc": "raw",
    "role": "unknown",
    "description": "topology_alt",
    "rep_start": 300,
    "replications": 25,
    "n_permutations": 199,
    "candidates": [
      "PointMMD-Gaussian",
      "PointMMD-Gaussian-median",
      "EnergyDistance",
      "FriedmanRafsky-MST",
      "Schilling-kNN-k1",
      "RawBlockMMD",
      "SC-B",
      "HybridBlockMMD-a0.50",
      "Schilling-kNN-k5",
      "Schilling-kNN-k10",
      "SlicedWasserstein",
      "ClassifierTwoSampleTest-logistic",
      "ClassifierTwoSampleTest-rf",
      "SC-A-Block",
      "Rosenbaum-CrossMatch"
    ],
    "predicted_seconds": 150.0
  }
]

SHARD_OUT_DIR = "results/phase6_highdim_shards"
os.makedirs(SHARD_OUT_DIR, exist_ok=True)

t0 = time.time()
for idx, task in enumerate(TASKS):
    print(f"\n=== Task {idx+1}/{len(TASKS)} {task['task_id']} ===")
    cell = Cell(family=task["family"], n0=task["n0"], n1=task["n1"], m=task["m"], d=task["d"], preproc=task["preproc"], role=task["role"], description=task["description"])
    out = os.path.join(SHARD_OUT_DIR, f"phase6_highdim_{cell.cell_id}_rep{task['rep_start']}_{task['rep_start']+task['replications']-1}.parquet")
    # Checkpoint every 25 reps if task larger than 25 — run_shard does it internally via single file
    run_shard(cell, rep_start=task["rep_start"], replications=task["replications"], n_permutations=task["n_permutations"], candidates=task["candidates"], workers=N_WORKERS, cache_dir=None, output=out)
    # Try download
    try:
        from google.colab import files
        files.download(out)
        print("Downloaded:", out)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)
    print(f"--- {time.time()-t0:.0f}s elapsed, {idx+1}/{len(TASKS)} tasks done ---")

print(f"Shard {SHARD_ID} done in {time.time()-t0:.0f}s")
print("Files kept in", SHARD_OUT_DIR)
